In [4]:
import os
import json
import pandas as pd

In [ ]:
test_root_dir = "Training/[원천]성취수준데이터셋_train/3학년"
test_log = []

for dir_A in os.listdir(test_root_dir): 
    dir_A_path = os.path.join(test_root_dir, dir_A)
    for sub_dir in os.listdir(dir_A_path):
        #sub_dir --> 실력평가n
        sub_dir_path = os.path.join(dir_A_path, sub_dir)
        # 실력평가 하위폴더가 '문항정오답표'로 끝나면
        if sub_dir_path.endswith('1_문항정오답표'):# 실력평가 하위폴더가 '문항정오답표'로 끝나면
            for json_file in os.listdir(sub_dir_path):
                json_path = os.path.join(sub_dir_path, json_file)            
                with open(json_path, 'r') as f:
                    json_content = json.load(f)
                    test_log.append(json_content)

In [ ]:
valid_root_dir = "Validation/[원천]성취수준데이터셋_valid/3학년"
valid_log = []

for dir_A in os.listdir(valid_root_dir): 
    dir_A_path = os.path.join(valid_root_dir, dir_A)
    for sub_dir in os.listdir(dir_A_path):
        sub_dir_path = os.path.join(dir_A_path, sub_dir)
        if sub_dir_path.endswith('1_문항정오답표'):
            for json_file in os.listdir(sub_dir_path):
                json_path = os.path.join(sub_dir_path, json_file)            
                with open(json_path, 'r') as f:
                    json_content = json.load(f)
                    valid_log.append(json_content)

In [25]:
problem_root_dir = "Training/[원천]성취수준데이터셋_train/3학년"
problem = []

for dir_A in os.listdir(problem_root_dir): 
    dir_A_path = os.path.join(problem_root_dir, dir_A)
    for sub_dir in os.listdir(dir_A_path):
        sub_dir_path = os.path.join(dir_A_path, sub_dir)
        if sub_dir_path.endswith('2_문항IRT'):
            for json_file in os.listdir(sub_dir_path):
                json_path = os.path.join(sub_dir_path, json_file)            
                with open(json_path, 'r') as f:
                    json_content = json.load(f)
                    problem.append(json_content)

In [26]:
test_log_df = pd.DataFrame(test_log)
valid_log_df = pd.DataFrame(valid_log)
problem_df = pd.DataFrame(problem)

In [27]:
icecream_df = pd.concat([test_log_df, valid_log_df])

# 필요한 칼럼만 추출
ice_df = icecream_df[['learnerID','testID','assessmentItemID','answerCode','Timestamp']]
pro_df = problem_df[['testID','assessmentItemID','knowledgeTag']]

icecream = pd.merge(ice_df, pro_df, on=['testID','assessmentItemID'], how='inner')

In [ ]:
icecream.rename(columns={'knowledgeTag':'skill_ID','learnerID':'user_id','answerCode':'correct'},inplace=True)

icecream = icecream.groupby('user_id').filter(lambda q: len(q) > 5).copy()
icecream = icecream.groupby('skill_ID').filter(lambda q: len(q) > 1).copy()

icecream["skill_id"], skill_labels = pd.factorize(icecream["skill_ID"])
icecream["skill_id"] += 1  

icecream['correct'] = icecream['correct'].astype(int)

,user_id,testID,assessmentItemID,correct,Timestamp,skill_ID,skill_id
0,A030000511,A030000094,A030094003,1,2020-03-10 08:42:01,461,1
1,A030000247,A030000094,A030094001,1,2020-08-26 06:03:34,461,1
2,A030000492,A030000094,A030094001,1,2020-07-28 06:16:04,461,1


In [ ]:
with open("[라벨]수학 지식체계 데이터 세트_210611.json", "r", encoding="utf-8") as f:
    kc_map = json.load(f)

from_id = []
to_id = []
for key, value in kc_map.items():
    from_concept = value["fromConcept"]
    to_concept = value["toConcept"]
    
    from_id.append({
        "kc_id": from_concept["id"],
        "kc_name": from_concept["name"],
        "semester": from_concept["semester"],
        "description": from_concept["description"],
        "chapter": from_concept["chapter"]["name"],
        #"achievement": from_concept["achievement"]["name"]
    })

    to_id.append({    
        "kc_id": to_concept["id"],
        "kc_name": to_concept["name"],
        "semester": to_concept["semester"],
        "description": to_concept["description"],
        "chapter": to_concept["chapter"]["name"],
        #"achievement": to_concept["achievement"]["name"]
    })


from_id = pd.DataFrame(from_id)
to_id = pd.DataFrame(to_id)

kc_map_df = pd.concat([from_id, to_id])

In [34]:
kc_map_df = kc_map_df.drop_duplicates()

In [ ]:
icecream['kc_id'] = icecream['skill_ID'].astype(int)
data = icecream.merge(kc_map_df, on='kc_id', how='inner')

data = data[['user_id','skill_id','correct','Timestamp','kc_name','semester']].sort_values(['user_id','Timestamp'])

# kc_name, semester 칼럼 불필요

data.to_csv("icecream_3rd.csv", encoding='utf-8-sig', index=False)